In [ ]:
# ============ VESUVIUS 3D v4 FAST ============
# 4 samples/file = ~3h/epoch
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import tifffile

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus: tf.config.experimental.set_memory_growth(g, True)

print('✅ TF:', tf.__version__, '| GPU:', len(gpus))

In [ ]:
# ============ CONFIG ============
TRAIN_IMAGES = './train_images'
TRAIN_LABELS = './train_labels'
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

VOL = 96
NC = 3
BS = 2
LR = 3e-4
EPOCHS = 100

# Sadece 4 köşe = hızlı
print(f'✅ 3D v4 FAST: {VOL}³, 4 samples/file')

In [ ]:
# ============ FAST GENERATOR ============
class Gen3D(keras.utils.Sequence):
    def __init__(self, img_dir, lbl_dir, files, aug=False):
        self.img_dir, self.lbl_dir = img_dir, lbl_dir
        self.files = list(files)
        self.aug = aug
        
        # 4 diagonal corners only (faster)
        self.corners = [(0,0,0), (0,1,1), (1,0,1), (1,1,0)]
        self.samples = [(i, c) for i in range(len(files)) for c in self.corners]
        
        print(f'  {len(self.samples)} samples ({len(self.samples)//BS} batches)')
        if aug: np.random.shuffle(self.samples)
    
    def __len__(self): return len(self.samples) // BS
    
    def get_pos(self, dim, c): return 0 if c == 0 else max(0, dim - VOL)
    
    def get_patch(self, vol, corner):
        d, h, w = vol.shape
        z, y, x = self.get_pos(d, corner[0]), self.get_pos(h, corner[1]), self.get_pos(w, corner[2])
        p = vol[z:z+VOL, y:y+VOL, x:x+VOL]
        if p.shape != (VOL, VOL, VOL):
            out = np.zeros((VOL, VOL, VOL), p.dtype)
            out[:p.shape[0], :p.shape[1], :p.shape[2]] = p
            return out
        return p
    
    def __getitem__(self, idx):
        batch = self.samples[idx*BS:(idx+1)*BS]
        X, Y, cache = [], [], {}
        
        for fidx, corner in batch:
            if fidx not in cache:
                v = tifffile.imread(os.path.join(self.img_dir, self.files[fidx])).astype(np.float32)
                nz = v > 0
                if nz.sum(): v = (v - v[nz].mean()) / (v[nz].std() + 1e-8)
                lp = os.path.join(self.lbl_dir, self.files[fidx])
                l = tifffile.imread(lp).astype(np.int32) if os.path.exists(lp) else np.zeros_like(v, np.int32)
                cache[fidx] = (v, l)
            
            v, l = cache[fidx]
            vp = self.get_patch(v, corner)
            lp = self.get_patch(l.astype(np.float32), corner).astype(np.int32)
            
            if self.aug:
                for ax in range(3):
                    if np.random.rand() > 0.5: vp, lp = np.flip(vp, ax), np.flip(lp, ax)
            
            oh = np.zeros((VOL, VOL, VOL, NC), np.float32)
            for c in range(NC): oh[..., c] = (lp == c)
            X.append(vp[..., np.newaxis].copy()); Y.append(oh.copy())
        
        return np.array(X, np.float32), np.array(Y, np.float32)
    
    def on_epoch_end(self):
        if self.aug: np.random.shuffle(self.samples)

print('✅ Generator')

In [ ]:
files = sorted([f for f in os.listdir(TRAIN_IMAGES) if f.endswith('.tif')])
np.random.seed(42); np.random.shuffle(files)
s = int(len(files) * 0.9)
train_f, val_f = files[:s], files[s:]
print(f'Train: {len(train_f)}×4={len(train_f)*4}')
print(f'Val: {len(val_f)}×4={len(val_f)*4}')

train_gen = Gen3D(TRAIN_IMAGES, TRAIN_LABELS, train_f, True)
val_gen = Gen3D(TRAIN_IMAGES, TRAIN_LABELS, val_f, False)

In [ ]:
X, Y = train_gen[0]
print(f'X:{X.shape} Y:{Y.shape} surf:{Y[...,1].mean()*100:.2f}%')

In [ ]:
def cb(x, f):
    x = layers.Conv3D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv3D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); return layers.ReLU()(x)

def unet(s, nc):
    i = layers.Input(s)
    c1 = cb(i, 32); p1 = layers.MaxPooling3D(2)(c1)
    c2 = cb(p1, 64); p2 = layers.MaxPooling3D(2)(c2)
    c3 = cb(p2, 128); p3 = layers.MaxPooling3D(2)(c3)
    c4 = cb(p3, 256)
    u5 = layers.Concatenate()([layers.Conv3DTranspose(128, 2, 2, 'same')(c4), c3]); c5 = cb(u5, 128)
    u6 = layers.Concatenate()([layers.Conv3DTranspose(64, 2, 2, 'same')(c5), c2]); c6 = cb(u6, 64)
    u7 = layers.Concatenate()([layers.Conv3DTranspose(32, 2, 2, 'same')(c6), c1]); c7 = cb(u7, 32)
    return Model(i, layers.Conv3D(nc, 1, activation='softmax')(c7))

model = unet((VOL,)*3 + (1,), NC)
print(f'✅ {model.count_params():,} params')

In [ ]:
def dice(yt, yp):
    yt, yp = tf.reshape(yt, [-1, NC]), tf.reshape(yp, [-1, NC])
    return 1 - tf.reduce_mean((2*tf.reduce_sum(yt*yp, 0) + 1e-6) / (tf.reduce_sum(yt+yp, 0) + 1e-6))

def loss(yt, yp): return tf.reduce_mean(keras.losses.categorical_crossentropy(yt, yp)) + dice(yt, yp)

def iou(yt, yp):
    t, p = yt[..., 1], tf.cast(yp[..., 1] > 0.5, tf.float32)
    return (tf.reduce_sum(t*p) + 1e-6) / (tf.reduce_sum(t) + tf.reduce_sum(p) - tf.reduce_sum(t*p) + 1e-6)

model.compile(keras.optimizers.Adam(LR), loss, [iou])
print('✅ Compiled')

In [ ]:
cbs = [
    keras.callbacks.ModelCheckpoint(f'{CHECKPOINT_DIR}/model_3d_v4_best.h5', 'val_iou', mode='max', save_best_only=True, save_weights_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau('val_iou', 0.5, 7, min_lr=1e-7, mode='max', verbose=1),
    keras.callbacks.EarlyStopping('val_iou', 20, mode='max', restore_best_weights=True, verbose=1)
]

print(f'🚀 {len(train_gen)} batches (~3h/epoch)')
h = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=cbs, workers=1)
print('🏆 Done!')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(h.history['loss'], 'b'); ax[0].plot(h.history['val_loss'], 'r'); ax[0].set_title('Loss')
ax[1].plot(h.history['iou'], 'b'); ax[1].plot(h.history['val_iou'], 'r'); ax[1].set_title('IoU')
plt.savefig('3d_v4.png'); plt.show()
print(f'Best: {max(h.history["val_iou"]):.4f}')